# Netflix Titles — SQL Analysis

**Week 3 Task: SQL Analysis**

This notebook loads the cleaned Netflix dataset into a local SQLite database and answers the same kinds of questions from Week 2's EDA — but in SQL. The 10 queries live in `schema_and_queries.sql` in this repo; they're also run here so results are visible without a separate SQL client.

**Why SQLite:** no server setup required, and the resulting `.sql` file is portable — the same schema and queries run on PostgreSQL or MySQL with only trivial syntax changes (noted inline where relevant).


In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os

pd.set_option('display.max_columns', None)


## 1. Rebuild the Cleaned Dataset

Same cleaning logic as Weeks 1–2.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-04-20/netflix_titles.csv"
df_raw = pd.read_csv(DATA_URL)

df = df_raw.drop_duplicates().copy()

text_cols = ['director', 'cast', 'country']
for col in text_cols:
    df[col] = df[col].replace(r'^\s*$', np.nan, regex=True)
    df[col] = df[col].fillna('Unknown')

df = df.dropna(subset=['date_added', 'rating']).copy()
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month

duration_num = df['duration'].str.extract(r'(\d+)').astype(float)
df['duration_minutes'] = np.where(df['type'] == 'Movie', duration_num[0], np.nan)
df['duration_seasons'] = np.where(df['type'] == 'TV Show', duration_num[0], np.nan)

print(f"Cleaned dataset ready: {df.shape[0]} rows")


## 2. Load Into a Normalized SQLite Database

The raw data stores `country` and `listed_in` (genre) as comma-separated lists inside a single column — that's not a relational structure, and it makes real JOINs impossible. So we normalize them into two junction tables:

- **`titles`** — one row per title (the "one" side)
- **`title_countries`** — one row per (title, country) pair (the "many" side)
- **`title_genres`** — one row per (title, genre) pair (the "many" side)

This mirrors a standard many-to-many relational design and is what makes the JOIN-based queries below meaningful rather than trivial.


In [ ]:
DB_PATH = "netflix.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.executescript("""
CREATE TABLE titles (
    show_id TEXT PRIMARY KEY,
    type TEXT NOT NULL,
    title TEXT NOT NULL,
    director TEXT,
    release_year INTEGER,
    rating TEXT,
    date_added TEXT,
    year_added INTEGER,
    month_added INTEGER,
    duration_minutes REAL,
    duration_seasons REAL
);

CREATE TABLE title_countries (
    show_id TEXT NOT NULL,
    country TEXT NOT NULL,
    FOREIGN KEY (show_id) REFERENCES titles(show_id)
);

CREATE TABLE title_genres (
    show_id TEXT NOT NULL,
    genre TEXT NOT NULL,
    FOREIGN KEY (show_id) REFERENCES titles(show_id)
);
""")

titles_cols = ['show_id','type','title','director','release_year','rating',
               'date_added','year_added','month_added','duration_minutes','duration_seasons']
titles_df = df[titles_cols].copy()
titles_df['date_added'] = titles_df['date_added'].astype(str)
titles_df.to_sql('titles', conn, if_exists='append', index=False)

countries_rows = [(r.show_id, c.strip()) for r in df.itertuples() for c in str(r.country).split(', ')]
pd.DataFrame(countries_rows, columns=['show_id', 'country']).to_sql('title_countries', conn, if_exists='append', index=False)

genres_rows = [(r.show_id, g.strip()) for r in df.itertuples() for g in str(r.listed_in).split(', ')]
pd.DataFrame(genres_rows, columns=['show_id', 'genre']).to_sql('title_genres', conn, if_exists='append', index=False)

conn.commit()
print("titles:", cur.execute("SELECT COUNT(*) FROM titles").fetchone()[0])
print("title_countries:", cur.execute("SELECT COUNT(*) FROM title_countries").fetchone()[0])
print("title_genres:", cur.execute("SELECT COUNT(*) FROM title_genres").fetchone()[0])


## 3. The 10 Analytical Queries

Each query answers a specific question and is verified against the equivalent pandas result from Week 2 where applicable.


### Q1. How many titles are Movies vs. TV Shows?
*(aggregation + GROUP BY — verify against Week 2 §2.1)*

In [ ]:
q1 = """
SELECT type, COUNT(*) AS title_count
FROM titles
GROUP BY type
ORDER BY title_count DESC;
"""
pd.read_sql_query(q1, conn)


### Q2. What are the top 10 countries by number of titles?
*(JOIN + GROUP BY + ORDER BY + LIMIT — verify against Week 2 §2.4)*

In [ ]:
q2 = """
SELECT tc.country, COUNT(*) AS title_count
FROM title_countries tc
JOIN titles t ON t.show_id = tc.show_id
WHERE tc.country != 'Unknown'
GROUP BY tc.country
ORDER BY title_count DESC
LIMIT 10;
"""
pd.read_sql_query(q2, conn)


### Q3. What are the top 10 genres by number of titles?
*(JOIN + GROUP BY + ORDER BY + LIMIT — verify against Week 2 §2.5)*

In [ ]:
q3 = """
SELECT tg.genre, COUNT(*) AS title_count
FROM title_genres tg
JOIN titles t ON t.show_id = tg.show_id
GROUP BY tg.genre
ORDER BY title_count DESC
LIMIT 10;
"""
pd.read_sql_query(q3, conn)


### Q4. What is the average movie duration by decade of release?
*(GROUP BY on a computed column)*

In [ ]:
q4 = """
SELECT (release_year / 10) * 10 AS decade,
       ROUND(AVG(duration_minutes), 1) AS avg_duration_minutes,
       COUNT(*) AS movie_count
FROM titles
WHERE type = 'Movie'
GROUP BY decade
ORDER BY decade;
"""
pd.read_sql_query(q4, conn)


### Q5. How many titles were added to Netflix each year, split by type?
*(GROUP BY multiple columns — verify against Week 2 §2.3)*

In [ ]:
q5 = """
SELECT year_added, type, COUNT(*) AS titles_added
FROM titles
WHERE year_added IS NOT NULL
GROUP BY year_added, type
ORDER BY year_added, type;
"""
pd.read_sql_query(q5, conn)


### Q6. Who are the top 10 directors by number of titles (excluding Unknown)?
*(WHERE + GROUP BY + ORDER BY + LIMIT)*

In [ ]:
q6 = """
SELECT director, COUNT(*) AS title_count
FROM titles
WHERE director != 'Unknown'
GROUP BY director
ORDER BY title_count DESC
LIMIT 10;
"""
pd.read_sql_query(q6, conn)


### Q7. Which countries have more TV shows than movies in the catalog?
*(JOIN + conditional aggregation + HAVING)*

In [ ]:
q7 = """
SELECT tc.country,
       SUM(CASE WHEN t.type = 'Movie' THEN 1 ELSE 0 END) AS movie_count,
       SUM(CASE WHEN t.type = 'TV Show' THEN 1 ELSE 0 END) AS tv_show_count
FROM title_countries tc
JOIN titles t ON t.show_id = tc.show_id
WHERE tc.country != 'Unknown'
GROUP BY tc.country
HAVING tv_show_count > movie_count
ORDER BY tv_show_count DESC;
"""
pd.read_sql_query(q7, conn)


### Q8. Which movies run longer than the overall average movie duration?
*(subquery in WHERE)*

In [ ]:
q8 = """
SELECT title, duration_minutes
FROM titles
WHERE type = 'Movie'
  AND duration_minutes > (
      SELECT AVG(duration_minutes) FROM titles WHERE type = 'Movie'
  )
ORDER BY duration_minutes DESC
LIMIT 15;
"""
pd.read_sql_query(q8, conn)


### Q9. For each of the top 5 countries by title count, what is their single most common genre?
*(CTE + window-function subquery — the "top-N per group" pattern)*

In [ ]:
q9 = """
WITH top_countries AS (
    SELECT tc.country, COUNT(*) AS title_count
    FROM title_countries tc
    JOIN titles t ON t.show_id = tc.show_id
    WHERE tc.country != 'Unknown'
    GROUP BY tc.country
    ORDER BY title_count DESC
    LIMIT 5
),
country_genre_counts AS (
    SELECT tc.country, tg.genre, COUNT(*) AS genre_count,
           RANK() OVER (PARTITION BY tc.country ORDER BY COUNT(*) DESC) AS genre_rank
    FROM title_countries tc
    JOIN title_genres tg ON tg.show_id = tc.show_id
    WHERE tc.country IN (SELECT country FROM top_countries)
    GROUP BY tc.country, tg.genre
)
SELECT country, genre, genre_count
FROM country_genre_counts
WHERE genre_rank = 1
ORDER BY genre_count DESC;
"""
pd.read_sql_query(q9, conn)


### Q10. Which directors (excluding Unknown) have worked across more than 2 different genres?
*(JOIN + GROUP BY + HAVING COUNT DISTINCT)*

In [ ]:
q10 = """
SELECT t.director, COUNT(DISTINCT tg.genre) AS distinct_genres
FROM titles t
JOIN title_genres tg ON tg.show_id = t.show_id
WHERE t.director != 'Unknown'
GROUP BY t.director
HAVING distinct_genres > 2
ORDER BY distinct_genres DESC
LIMIT 15;
"""
pd.read_sql_query(q10, conn)


In [ ]:
conn.close()


## 4. Notes on Verification

Q1, Q2, Q3, and Q5 were cross-checked against the equivalent pandas `.value_counts()` / `.groupby()` results from Week 2's EDA notebook — the row counts and rankings match exactly, confirming the SQL layer and the pandas layer agree on the same underlying cleaned data.
